# Aircraft Anomaly Detection Model

End-to-end ML pipeline for aircraft predictive maintenance:
- **Binary Classification** — Detect anomalous flights from operational metrics (GBT)
- **Multi-class Classification** — Identify specific anomaly types (Random Forest)
- **Unsupervised Detection** — Isolation Forest on raw sensor telemetry
- **Maintenance Prediction** — Composite risk scoring for fleet prioritization

Data: `genie_zeroops_mfg_catalog.default` — gold & silver medallion layers

In [0]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, f1_score, accuracy_score, precision_score, recall_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

CATALOG = "genie_zeroops_mfg_catalog"
SCHEMA = "default"

# Load gold tables
flight_df = spark.read.table(f"{CATALOG}.{SCHEMA}.gold_flight_summary").toPandas()
health_df = spark.read.table(f"{CATALOG}.{SCHEMA}.gold_aircraft_health").toPandas()

# Feature columns (anomaly_record_count excluded — target leakage)
FEATURE_COLS = [
    "flight_duration_hours", "max_altitude", "avg_cruise_speed",
    "total_fuel_burned", "max_egt_eng1", "max_egt_eng2",
    "max_vibration", "avg_oil_pressure_eng1", "avg_oil_pressure_eng2",
    "min_hydraulic_pressure",
]

print(f"Flight summary: {flight_df.shape[0]} flights, {flight_df.shape[1]} columns")
print(f"Aircraft health: {health_df.shape[0]} aircraft")
display(flight_df.head(5))

In [0]:
# Ensure has_anomaly is proper boolean (came as object from Spark)
flight_df["has_anomaly"] = flight_df["has_anomaly"].fillna(False).astype(bool)

# --- Target distribution ---
anomaly_rate = flight_df["has_anomaly"].mean()
print(f"Anomaly rate: {anomaly_rate:.1%} ({flight_df['has_anomaly'].sum()} / {len(flight_df)})")
print(f"\nClass distribution:\n{flight_df['has_anomaly'].value_counts()}\n")

# --- Anomaly type breakdown ---
print("Anomaly types among flagged flights:")
display(flight_df.loc[flight_df["has_anomaly"], "anomaly_type"].value_counts().to_frame("count"))

# --- Target leakage check ---
all_candidates = FEATURE_COLS + ["anomaly_record_count"]
corr_target = flight_df[all_candidates].corrwith(
    flight_df["has_anomaly"].astype(int)
).sort_values(ascending=False)
print("\nCorrelation with has_anomaly:")
display(corr_target.to_frame("correlation"))
print("\u26a0\ufe0f anomaly_record_count correlates perfectly with target \u2014 excluded as leakage.")

# --- Visualizations ---
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

flight_df["has_anomaly"].value_counts().plot(
    kind="bar", ax=axes[0], color=["steelblue", "tomato"]
)
axes[0].set_title("Anomaly Class Distribution")
axes[0].set_xticklabels(["Normal", "Anomaly"], rotation=0)

corr = flight_df[FEATURE_COLS].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=axes[1],
            vmin=-1, vmax=1, annot_kws={"size": 7})
axes[1].set_title("Feature Correlations")

means = flight_df.groupby("has_anomaly")[FEATURE_COLS].mean().T
means.columns = ["Normal", "Anomaly"]
diff_pct = ((means["Anomaly"] - means["Normal"]) / means["Normal"].replace(0, np.nan) * 100).sort_values()
colors = ["crimson" if v > 0 else "steelblue" for v in diff_pct]
diff_pct.plot(kind="barh", ax=axes[2], color=colors)
axes[2].set_title("Mean % Difference (Anomaly vs Normal)")
axes[2].set_xlabel("% Difference")

plt.tight_layout()
plt.show()

In [0]:
# Class imbalance detected: ~10.3% anomaly rate. Using balanced sample weights.

X = flight_df[FEATURE_COLS].copy()
y = flight_df["has_anomaly"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} ({y_train.mean():.1%} positive) | Test: {len(X_test)} ({y_test.mean():.1%} positive)")

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
    ]), FEATURE_COLS)
])

binary_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42
    ))
])

sample_weights = compute_sample_weight("balanced", y_train)

mlflow.set_experiment(f"/Users/ayman.el-ghazali@databricks.com/FE Bar/Aircraft Anomaly Detection")

with mlflow.start_run(run_name="binary_anomaly_gbt") as run:
    binary_pipeline.fit(X_train, y_train, classifier__sample_weight=sample_weights)

    y_pred = binary_pipeline.predict(X_test)
    y_proba = binary_pipeline.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_val = roc_auc_score(y_test, y_proba)

    mlflow.log_params({"model_type": "GradientBoostingClassifier",
                       "n_estimators": 200, "max_depth": 4, "learning_rate": 0.1,
                       "class_weighting": "balanced_sample_weight", "n_features": len(FEATURE_COLS)})
    mlflow.log_metrics({"accuracy": acc, "f1_score": f1, "roc_auc": auc_val})

    sig = infer_signature(X_train.head(100), binary_pipeline.predict(X_train.head(100)))
    binary_model_info = mlflow.sklearn.log_model(
        binary_pipeline, name="binary_anomaly_model",
        signature=sig, input_example=X_train.head(3)
    )
    binary_run_id = run.info.run_id

print(f"\nAccuracy: {acc:.3f} | F1: {f1:.3f} | ROC-AUC: {auc_val:.3f}\n")
print(classification_report(y_test, y_pred, target_names=["Normal", "Anomaly"]))

# --- Evaluation plots ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"])
axes[0].set_title("Confusion Matrix"); axes[0].set_ylabel("Actual"); axes[0].set_xlabel("Predicted")

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, "b-", label=f"AUC = {auc_val:.3f}")
axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set_title("ROC Curve"); axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR"); axes[1].legend()

feat_imp = pd.Series(
    binary_pipeline.named_steps["classifier"].feature_importances_, index=FEATURE_COLS
).sort_values()
feat_imp.plot(kind="barh", ax=axes[2], color="steelblue")
axes[2].set_title("Feature Importance (Binary Model)")

plt.tight_layout()
mlflow.log_figure(fig, "binary_evaluation.png")
plt.show()

In [0]:
mlflow.end_run()  # ensure no stale run from prior cell

# Classify specific anomaly types among anomalous flights only
anomaly_flights = flight_df[flight_df["has_anomaly"]].copy()
print(f"Anomalous flights: {len(anomaly_flights)} | Anomaly types: {anomaly_flights['anomaly_type'].nunique()}")
print(anomaly_flights["anomaly_type"].value_counts())

X_mc = anomaly_flights[FEATURE_COLS].copy()
y_mc = anomaly_flights["anomaly_type"]

try:
    X_mc_train, X_mc_test, y_mc_train, y_mc_test = train_test_split(
        X_mc, y_mc, test_size=0.25, random_state=42, stratify=y_mc
    )
except ValueError:
    X_mc_train, X_mc_test, y_mc_train, y_mc_test = train_test_split(
        X_mc, y_mc, test_size=0.25, random_state=42
    )
    print("\u26a0\ufe0f Stratified split failed (rare class) \u2014 using random split")

mc_pipeline = Pipeline([
    ("preprocessor", ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
        ]), FEATURE_COLS)
    ])),
    ("classifier", RandomForestClassifier(
        n_estimators=300, max_depth=8, random_state=42,
        class_weight="balanced_subsample"
    ))
])

with mlflow.start_run(run_name="multiclass_anomaly_type") as mc_run:
    mc_pipeline.fit(X_mc_train, y_mc_train)
    y_mc_pred = mc_pipeline.predict(X_mc_test)

    mc_acc = accuracy_score(y_mc_test, y_mc_pred)
    mc_f1 = f1_score(y_mc_test, y_mc_pred, average="macro")

    mlflow.log_params({"model_type": "RandomForestClassifier", "n_estimators": 300,
                       "n_classes": y_mc.nunique(), "training_size": len(X_mc_train)})
    mlflow.log_metrics({"accuracy": mc_acc, "macro_f1": mc_f1})

    sig_mc = infer_signature(X_mc_train.head(20), mc_pipeline.predict(X_mc_train.head(20)))
    mc_model_info = mlflow.sklearn.log_model(
        mc_pipeline, name="multiclass_anomaly_model",
        signature=sig_mc, input_example=X_mc_train.head(3)
    )

print(f"\nMulti-class Accuracy: {mc_acc:.3f} | Macro-F1: {mc_f1:.3f}\n")
print(classification_report(y_mc_test, y_mc_pred))

fig, ax = plt.subplots(figsize=(10, 8))
cm_mc = confusion_matrix(y_mc_test, y_mc_pred, labels=mc_pipeline.classes_)
sns.heatmap(cm_mc, annot=True, fmt="d", cmap="Oranges", ax=ax,
            xticklabels=mc_pipeline.classes_, yticklabels=mc_pipeline.classes_)
ax.set_title("Anomaly Type Confusion Matrix")
ax.set_ylabel("Actual"); ax.set_xlabel("Predicted")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [0]:
# Sample 100K records from silver telemetry with key sensor features
SENSOR_COLS = [
    "engine_avg_egt", "engine_avg_vibration", "fuel_burn_rate", "hydraulic_health_index",
    "egt_eng1", "egt_eng2", "oil_pressure_eng1", "oil_pressure_eng2",
    "vibration_n1_eng1", "vibration_n1_eng2", "n1_speed_eng1", "n1_speed_eng2",
    "hyd_press_sys1", "hyd_press_sys2", "hyd_press_sys3",
    "altitude", "indicated_airspeed", "fuel_flow_eng1", "fuel_flow_eng2",
    "battery_voltage",
]
META_COLS = ["flight_id", "anomaly_type", "flight_phase"]

telemetry_sample = (
    spark.read.table(f"{CATALOG}.{SCHEMA}.silver_enriched_telemetry")
    .select(META_COLS + SENSOR_COLS)
    .sample(0.1, seed=42)
    .limit(100000)
    .toPandas()
)
print(f"Telemetry sample: {telemetry_sample.shape}")

X_iso = telemetry_sample[SENSOR_COLS].dropna()
valid_idx = X_iso.index
print(f"Records after dropna: {len(X_iso)}")

scaler_iso = StandardScaler()
X_iso_scaled = scaler_iso.fit_transform(X_iso)

iso_forest = IsolationForest(n_estimators=200, contamination=0.10, random_state=42, n_jobs=-1)
iso_pred = iso_forest.fit_predict(X_iso_scaled)
iso_scores = iso_forest.decision_function(X_iso_scaled)

# Compare with actual labels
result = telemetry_sample.loc[valid_idx].copy()
result["iso_anomaly"] = (iso_pred == -1).astype(int)
result["iso_score"] = iso_scores
result["actual_anomaly"] = (result["anomaly_type"].notna() & (result["anomaly_type"] != "")).astype(int)

iso_prec = precision_score(result["actual_anomaly"], result["iso_anomaly"])
iso_rec = recall_score(result["actual_anomaly"], result["iso_anomaly"])
iso_f1 = f1_score(result["actual_anomaly"], result["iso_anomaly"])

print(f"\nIsolation Forest vs. Labeled Anomalies:")
print(f"  Precision: {iso_prec:.3f} | Recall: {iso_rec:.3f} | F1: {iso_f1:.3f}")
print(f"  Detected rate: {result['iso_anomaly'].mean():.1%} | Actual rate: {result['actual_anomaly'].mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(iso_scores[result["actual_anomaly"] == 0], bins=50, alpha=0.6, label="Normal", color="steelblue")
axes[0].hist(iso_scores[result["actual_anomaly"] == 1], bins=50, alpha=0.6, label="Anomaly", color="tomato")
axes[0].set_title("Isolation Forest Anomaly Score Distribution")
axes[0].set_xlabel("Score (lower = more anomalous)"); axes[0].legend()

cm_iso = confusion_matrix(result["actual_anomaly"], result["iso_anomaly"])
sns.heatmap(cm_iso, annot=True, fmt="d", cmap="Purples", ax=axes[1],
            xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"])
axes[1].set_title("Isolation Forest vs. Actual Labels")
axes[1].set_ylabel("Actual"); axes[1].set_xlabel("Predicted")

plt.tight_layout()
plt.show()

In [0]:
# Score each aircraft using its latest flight features + health metrics
latest_flights = (
    flight_df.sort_values("departure_time", ascending=False)
    .groupby("tail_number")
    .first()
    .reset_index()
)

latest_features = latest_flights[FEATURE_COLS].copy()
anomaly_proba = binary_pipeline.predict_proba(latest_features)[:, 1]
latest_flights["ml_anomaly_probability"] = anomaly_proba

# Merge with aircraft health
predictions = latest_flights[["tail_number", "airline", "aircraft_type", "ml_anomaly_probability"]].merge(
    health_df[["tail_number", "health_risk_score", "anomaly_rate",
               "days_since_last_maintenance", "days_until_next_maintenance",
               "total_flights", "anomaly_flight_count"]],
    on="tail_number", how="left"
)

# Composite maintenance priority (0–100)
predictions["maintenance_priority_score"] = (
    predictions["ml_anomaly_probability"] * 35
    + predictions["health_risk_score"].fillna(0) * 0.35
    + np.where(predictions["days_until_next_maintenance"].fillna(365) < 30, 30,
               np.where(predictions["days_until_next_maintenance"].fillna(365) < 90, 15, 0))
).clip(0, 100)

predictions["risk_category"] = pd.cut(
    predictions["maintenance_priority_score"],
    bins=[-0.1, 20, 40, 60, 100],
    labels=["LOW", "MEDIUM", "HIGH", "CRITICAL"]
)
predictions = predictions.sort_values("maintenance_priority_score", ascending=False)

print(f"Maintenance predictions for {len(predictions)} aircraft\n")
print(f"Risk distribution:\n{predictions['risk_category'].value_counts().sort_index()}")
display(predictions.head(20))

# Write to catalog
pred_sdf = spark.createDataFrame(predictions)
pred_sdf.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.ml_maintenance_predictions")
print(f"\n\u2705 Written to {CATALOG}.{SCHEMA}.ml_maintenance_predictions")

In [0]:
import mlflow
mlflow.set_registry_uri("databricks-uc")

# Register binary classification model
binary_model_name = f"{CATALOG}.{SCHEMA}.aircraft_anomaly_binary_classifier"
reg_binary = mlflow.register_model(model_uri=binary_model_info.model_uri, name=binary_model_name)
print(f"\u2705 Binary model registered: {binary_model_name} v{reg_binary.version}")

# Register multi-class model
mc_model_name = f"{CATALOG}.{SCHEMA}.aircraft_anomaly_type_classifier"
reg_mc = mlflow.register_model(model_uri=mc_model_info.model_uri, name=mc_model_name)
print(f"\u2705 Multi-class model registered: {mc_model_name} v{reg_mc.version}")

print("\n--- Summary ---")
print(f"  Binary model:      {binary_model_name}")
print(f"  Multi-class model: {mc_model_name}")
print(f"  Predictions table: {CATALOG}.{SCHEMA}.ml_maintenance_predictions")
print(f"  MLflow experiment: /Users/ayman.el-ghazali@databricks.com/FE Bar/Aircraft Anomaly Detection")